# CMR Compare

CMR baseline with serial position scaling for CRU comparison


CMR Compare is a baseline CMR model designed for direct comparison with CRU models. It incorporates **serial position scaling** of encoding drift (like CRU) but **without the item identification mechanism**, allowing researchers to isolate the contribution of each component.

## Purpose

When comparing CRU and CMR architectures, we need to control for differences unrelated to the core theoretical distinctions. CMR Compare:

1. Uses the same serial position scaling as CRU models
2. Maintains perfect item identification (no slip matrix)
3. Uses standard CMR's item-independent stopping rule

This allows factorial comparisons of:
- Item identification: Present (CRU) vs Absent (CMR Compare)
- Termination: Context-based vs Item-independent

## Key Mechanism

### Serial Position Scaling

Like the Omnibus CRU-CMR, context drift decreases across positions:

$$\beta_i = \beta_{max} \cdot \beta_{dec}^{i-1}$$

This means early items receive stronger context integration than later items.

### No Item Identification

Unlike CRU models, recall output is deterministic—the item with highest activation is recalled directly without passing through a confusability-based identification stage.

## Parameters

| Parameter | Symbol | Description |
|-----------|--------|-------------|
| `encoding_drift_rate` | $\beta_{max}$ | Initial context drift rate |
| `encoding_drift_decrease` | $\beta_{dec}$ | Decay factor per position |
| `stop_probability_scale` | $\theta_s$ | Base stopping probability |
| `stop_probability_growth` | $\theta_r$ | Stopping growth rate |

Plus standard CMR parameters.

## Mathematical Specification

### Encoding

At position $i$:

$$\mathbf{c}^{IN}_i = M^{FC} \mathbf{f}_i$$

$$\mathbf{c}_{new} = \rho_i \mathbf{c}_{old} + \beta_i \cdot \mathbf{c}^{IN}_i$$

Where $\beta_i = \beta_{max} \cdot \beta_{dec}^{i-1}$.

### Retrieval

Standard CMR retrieval with item-independent stopping:

$$P(stop) = \theta_s \cdot e^{j \cdot \theta_r}$$

Where $j$ is the number of items already recalled.

## Usage

In [ ]:
from jaxcmr.models_cru_to_cmr.cmr_compare import CMR, CMRFactory

params = {
    # Standard CMR parameters
    "encoding_drift_rate": 0.8,
    "start_drift_rate": 0.3,
    "recall_drift_rate": 0.5,
    "learning_rate": 0.5,
    "primacy_scale": 2.0,
    "primacy_decay": 0.8,
    "shared_support": 0.05,
    "item_support": 0.25,
    "choice_sensitivity": 1.0,

    # Serial position scaling (like CRU)
    "encoding_drift_decrease": 0.95,

    # Item-independent stopping
    "stop_probability_scale": 0.01,
    "stop_probability_growth": 0.3,
}

factory = CMRFactory(dataset, features=None)
model = factory.create_model(params)

## Comparison: Model Variants

| Model | Serial Position Scaling | Item Identification | Termination |
|-------|------------------------|---------------------|-------------|
| Standard CMR | No | Perfect | Item-independent |
| **CMR Compare** | Yes | Perfect | Item-independent |
| CMR CompTerm | Yes | Perfect | Context-based |
| Omnibus CRU-CMR | Yes | Confusability | Item-independent |
| CompTerm CRU-CMR | Yes | Confusability | Context-based |

## Why Serial Position Scaling?

Empirical data from serial recall shows:
- **Primacy**: First items recalled more accurately
- **Recency**: Last items also recalled well
- **Middle items**: Most errors occur here

Standard CMR's primacy gradient (affecting MCF learning) captures some of this. Serial position scaling adds complementary effects on context encoding.

## Factorial Model Comparison

The CRU-to-CMR project uses these model variants for factorial comparison:

**2×2 Design:**

|  | Perfect ID | Confusability ID |
|--|-----------|-----------------|
| **Item-independent stop** | CMR Compare | Omnibus CRU-CMR |
| **Context-based stop** | CMR CompTerm | CompTerm CRU-CMR |

This design isolates:
1. **Item identification effect**: Compare left column to right column
2. **Termination mechanism effect**: Compare top row to bottom row
3. **Interaction**: Whether identification and termination interact

## Predictions

### Compared to Standard CMR

With serial position scaling:
- Stronger recency effect (later items encoded with lower drift)
- Context at end of list closer to final items
- May reduce forward asymmetry in lag-CRP

### Compared to Omnibus CRU

Without item identification:
- No transposition errors (confusion-based)
- Only omission errors possible
- May underestimate error rates in serial recall

## When to Use

Use CMR Compare when:
- You need a CMR baseline for CRU comparison
- You want to test serial position scaling without confusability
- You're conducting factorial model comparisons

For standalone CMR modeling without CRU comparison, the standard CMR model is typically preferred.